In [9]:
#imports
import pandas as pd
import numpy as np
import os
raw_file = pd.read_csv("D:/Data_CHU/predict_ttf1.csv")
raw_file


,id,Lame manquante,Type histologique,ttf1,Age,Sexe,Pièce op,TNM,Tabac,Atcd carcinologiqsues
0,25P13027,NaN,Adénocarcinome G3,+,77.0,F,PO,T1a N0,oui,cancer du sein
1,25P16493,NaN,Carcinoide typique,-,64.0,H,PO,T1c N0,Non,Non
2,25P16544,NaN,Pas de cancer,-,85.0,F,PO,NaN,NaN,NaN
3,25P17513,NaN,Adénocarcinome G2,+,52.0,H,PO,T1c N0,Oui,Non
4,25P18147,NaN,Adénocarcinome G3,+,80.0,F,PO,T3N0,Oui,"Vessie, Sein"
...,...,...,...,...,...,...,...,...,...,...
108,26P4222,NaN,NaN,+,NaN,NaN,NaN,NaN,NaN,NaN
109,26P4381,NaN,NaN,+,NaN,NaN,NaN,NaN,NaN,NaN
110,26P4575,NaN,NaN,+,NaN,NaN,NaN,NaN,NaN,NaN
111,26P4751,NaN,NaN,+,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
working_df = raw_file[['id','ttf1']]
working_df = working_df.drop_duplicates()
working_df['label'] = 

In [ ]:
working_df["label"] = working_df["ttf1"].str.contains(r"\+").astype(int)


,id,ttf1,label
0,25P13027,+,1
1,25P16493,-,0
2,25P16544,-,0
3,25P17513,+,1
4,25P18147,+,1
...,...,...,...
108,26P4222,+,1
109,26P4381,+,1
110,26P4575,+,1
111,26P4751,+,1


In [14]:
working_df = working_df[['id','label']]
working_df.to_csv('C:/Users/manon/Documents/pathology_MIL/data/splits_chu/data_chu.csv', index=False)

In [1]:
"""
Script de répartition des fichiers .h5 en dossiers train / validation / test
basé sur le fichier data_chu_split.csv

Version sécurisée pour fichiers HDF5 :
- copie au lieu de déplacement
- vérification taille fichier
- vérification ouverture HDF5
- gestion des erreurs
"""

import pandas as pd
import shutil
import argparse
from pathlib import Path
import os
import h5py


def verify_h5_file(path: Path) -> bool:
    """
    Vérifie qu'un fichier HDF5 peut être ouvert correctement.
    """
    try:
        with h5py.File(path, "r"):
            return True
    except Exception as e:
        print(f"  [CORRUPTED] {path.name} : {e}")
        return False


def split_files(csv_path: str, h5_dir: str, output_dir: str) -> None:
    df = pd.read_csv(csv_path)

    h5_dir = Path(h5_dir)
    output_dir = Path(output_dir)

    # Créer les sous-dossiers
    for folder in ["train", "validation", "test"]:
        (output_dir / folder).mkdir(parents=True, exist_ok=True)

    stats = {
        "train": 0,
        "validation": 0,
        "test": 0,
        "skipped": 0,
        "failed": 0,
    }

    for _, row in df.iterrows():
        file_id = row["slide_id"]
        split = row["set"]

        src = h5_dir / f"{file_id}.h5"

        if not src.exists():
            print(f"  [SKIP] {src.name} introuvable")
            stats["skipped"] += 1
            continue

        # Vérification source avant copie
        if not verify_h5_file(src):
            print(f"  [SKIP] {src.name} source corrompue")
            stats["failed"] += 1

            continue

        dst = output_dir / split / src.name

        try:
            # Copie avec métadonnées
            shutil.copy2(src, dst)

            # Vérification taille
            src_size = os.path.getsize(src)
            dst_size = os.path.getsize(dst)

            if src_size != dst_size:
                print(f"  [FAILED] {src.name} taille différente après copie")
                dst.unlink(missing_ok=True)
                stats["failed"] += 1
                continue

            # Vérification ouverture HDF5 destination
            if not verify_h5_file(dst):
                print(f"  [FAILED] {src.name} copie corrompue")
                dst.unlink(missing_ok=True)
                stats["failed"] += 1
                continue

            print(f"  [OK]   {src.name} → {split}/")
            stats[split] += 1

        except Exception as e:
            print(f"  [ERROR] {src.name} : {e}")
            stats["failed"] += 1

    # Résumé
    print("\n── Résumé ──────────────────────────────")
    print(f"  train      : {stats['train']} fichier(s)")
    print(f"  validation        : {stats['validation']} fichier(s)")
    print(f"  test       : {stats['test']} fichier(s)")
    print(f"  skippés    : {stats['skipped']} fichier(s)")
    print(f"  erreurs    : {stats['failed']} fichier(s)")
    print("────────────────────────────────────────")


csv_dir = "C:/Users/manon/Documents/pathology_MIL/data/splits_chu/data_chu_split.csv"

h5_dir_target = "C:/Users/manon/Documents/pathology_MIL/data/CHU_UNI2_embeds/IHC/"
h5_dir = "D:/Embeddings/Data_CHU/UNI2H_embeddings/IHC"

split_files(csv_dir, h5_dir, h5_dir_target)

  [OK]   25P13027.h5 → train/
  [OK]   25P16493.h5 → train/
  [OK]   25P16544.h5 → train/
  [OK]   25P17513.h5 → test/
  [OK]   25P18147.h5 → test/
  [OK]   25P7216.h5 → test/
  [OK]   25P9801.h5 → train/
  [OK]   25P9947.h5 → train/
  [OK]   25P11397.h5 → train/
  [OK]   25P11414.h5 → validation/
  [OK]   25P12995.h5 → train/
  [OK]   25P6168.h5 → train/
  [OK]   25P6193_1.h5 → train/
  [OK]   25P6193.h5 → train/
  [OK]   25P7615.h5 → train/
  [OK]   25P7883.h5 → train/
  [OK]   25P8416.h5 → train/
  [OK]   25P8589.h5 → train/
  [OK]   25P9546.h5 → test/
  [OK]   25P9546.h5 → train/
  [OK]   25P9599.h5 → train/
  [CORRUPTED] 25P15592.h5 : Unable to synchronously open file (bad object header version number)
  [SKIP] 25P15592.h5 source corrompue
  [OK]   25P16135.h5 → train/
  [OK]   25P16569.h5 → train/
  [OK]   25P18069.h5 → train/
  [OK]   25P18616.h5 → test/
  [OK]   25P18860.h5 → validation/
  [OK]   25P19032.h5 → test/
  [OK]   25P19078.h5 → train/
  [OK]   25P19219.h5 → train/
  